In [ ]:
"""
=========================================================
08_score_customers.py
=========================================================

Purpose
-------
Load the final trained model and score every customer
with a propensity score.

Input
-----
data/processed/feature_engineered.csv
models/final_catboost_model.cbm

Output
------
outputs/customer_propensity_scores.csv
"""

In [ ]:
import warnings
import pandas as pd

from catboost import CatBoostClassifier
from pathlib import Path
warnings.filterwarnings("ignore")

In [ ]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

In [ ]:
##########################################################
# LOAD DATA
##########################################################

print("=" * 70)
print("CUSTOMER PROPENSITY SCORING")
print("=" * 70)

df = pd.read_csv(f"{BASE_DIR}/data/processed/feature_engineered.csv")

In [ ]:
##########################################################
# FINAL FEATURES
##########################################################

FINAL_FEATURES = [

    "current_age",

    "vintage_months",

    "DP_Value_log",

    "Total_Margin_log",

    "ord_cash_log",

    "Category_Bucket_final",

    "self_dealer_status",

    "Trading_member_code",

    "plan",

    "recency",

    "never_traded"

]

In [ ]:
##########################################################
# CATEGORICAL FEATURES
##########################################################

categorical_columns = [

    "Category_Bucket_final",

    "self_dealer_status",

    "Trading_member_code",

    "plan"

]

In [ ]:
##########################################################
# PREPARE DATA
##########################################################

X = df[FINAL_FEATURES].copy()

for col in categorical_columns:

    X[col] = X[col].fillna("Missing").astype(str)

In [ ]:
##########################################################
# LOAD MODEL
##########################################################

model = CatBoostClassifier()

model.load_model("models/final_catboost_model.cbm")

In [ ]:
##########################################################
# SCORE CUSTOMERS
##########################################################

df["propensity_score"] = model.predict_proba(X)[:, 1]

In [ ]:
##########################################################
# SORT (Highest Propensity First)
##########################################################

df = df.sort_values(

    "propensity_score",

    ascending=False

)

In [ ]:
##########################################################
# KEEP RELEVANT COLUMNS
##########################################################

output = df[

    [

        "Client_Code",

        "propensity_score"

    ]

    + FINAL_FEATURES

].copy()

In [ ]:
##########################################################
# SAVE
##########################################################

output.to_csv(

    f"{BASE_DIR}/outputs/customer_propensity_scores.csv",

    index=False

)

In [ ]:
##########################################################
# SUMMARY
##########################################################

print("\nScoring Complete")

print("-" * 60)

print(f"Customers Scored : {len(output):,}")

print()

print(output["propensity_score"].describe())

print()

print("Top 10 Customers")

print(output.head(10))

print()

print("Saved to:")

print(f"{BASE_DIR}/outputs/customer_propensity_scores.csv")